# Pytorch Tutorial

## Basics: Tensor

In [ ]:
# 导入torch和torchvision库
import torch
import torchvision

### Initializing a tensor

In [ ]:
import torch
import numpy as np

data = [[1,2], [3,4]]
x_data = torch.tensor(data)  # 从列表创建 Tensor
print(x_data)

np_array = np.array(data)
x_np = torch.from_numpy(np_array)  # 将 NumPy 数组转为 Tensor
print(x_np)

x_ones = torch.ones_like(x_data)  # 创建与 x_data 同形状的全 1 Tensor
x_rand = torch.rand_like(x_data, dtype=torch.float)  # 同形状的随机浮点 Tensor，取值在 [0, 1)
print(x_rand)

# 按指定形状创建随机、全 1 和全 0 Tensor
shape =(2,3)
rand_tensor = torch.rand(shape)
ones_tensor =torch.ones(shape)
zeros_tensor= torch.zeros(shape)
print(rand_tensor)

### Attributes of a Tensor

In [ ]:
# 查看 Tensor 的形状、数据类型和所在设备
tensor = torch.rand(3,4)
print("Shape of tensor: {}".format(tensor.shape))
print("Datatype of tensor: {}".format(tensor.dtype))
print("Device tensor is stored on: {}".format(tensor.device))

### Device

In [ ]:
if torch.cuda.is_available():
    tensor = tensor.to('cuda')  # 将 Tensor 移到 GPU
else:
    print("cuda is not available!")
print("Device tensor is stored on: {}".format(tensor.device))

device = 0  # 指定使用的 GPU 编号
if torch.cuda.is_available():
    tensor = tensor.to('cuda:{}'.format(device))
print("Device tensor is stored on: {}".format(tensor.device))

tensor = tensor.cpu()  # 将 Tensor 移回 CPU
# tensor = tensor.to('cpu')
print("Device tensor is stored on: {}".format(tensor.device))


### Indexing, slicing and joining tensors

In [ ]:
l = list(range(9))
tensor =torch.tensor(l).reshape((3,3))  # 将 9 个元素排列成 3×3 矩阵
print("Original 3*3 tensor:")
print(tensor)

# 按行、列索引 Tensor
print('First row: ', tensor[0])
print('First column: ', tensor[:, 0])
print('Last column: ', tensor[...,-1])

t1= torch.cat([tensor, tensor], dim=1)  # 沿现有的列维度拼接
t2 = torch.stack([tensor,tensor] , dim=1)  # 新增一个维度，再将 Tensor 堆叠起来

print("\nAfter concat:")
print(t1)
print("\nAfter stack:")
print(t2)

### Arithmetic operations

In [ ]:
# 矩阵乘法的三种写法，.T 表示转置
y1 = tensor @ tensor.T
y2= tensor.matmul(tensor.T)
y3 = torch.matmul(tensor,tensor.T)
print('-'*20)
print(y1)
print(y2)
print(y3)

# 对应位置的元素相乘
z1 = tensor * tensor
z2 = tensor.mul(tensor)
z3 = torch.mul(tensor,tensor)
print('-'*20)
print(z1)
print(z2)
print(z3)

# 每个元素加 5 或乘 5，结果保存为新 Tensor
i1 = tensor.add(5)
i2 = tensor.mul(5)
print('-'*20)
print(i1)
print(i2)


###  🎯 TASK-1

1. Generate a 2x4 matrix t1_a with values \([1, 2, ..., 8]\) in row-major order.
2. Generate a 4x2 matrix t1_b with values \([1,1,1, ..., 1]\).
3. Multiply the two matrices and output the result as t1_c.

In [ ]:
t1_a = 
t1_b = 
t1_c = 

print(t1_a)
print(t1_b)
print(t1_c)

# Training a CNN Network

### 🎯 TASK-2 
Install tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

import time
from tqdm import tqdm

## Dataset

### Loading train set and test set

In [ ]:
# Compose 将多个预处理步骤按顺序组合起来
transform = transforms.Compose(
    [transforms.ToTensor(),  # 图像转为 [C, H, W] Tensor，像素值缩放到 [0, 1]
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])  # 将三个通道的像素值归一化到 [-1, 1]

batch_size = 32
data_path = "../data"
# 加载 CIFAR-10 数据集，并在读取图片时应用上述预处理
trainset = torchvision.datasets.CIFAR10(root=data_path, train=True,
                                        download=True, transform=transform)
# DataLoader 按批次提供数据；shuffle=True 让训练数据每轮打乱顺序
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)
 
testset = torchvision.datasets.CIFAR10(root=data_path, train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)
 
classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
 

### Check some samples

In [ ]:
def imshow(img):
    img = img / 2 + 0.5  # 将像素值还原到 [0, 1]
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg,(1, 2, 0)))  # 将 [C, H, W] 转成绘图使用的 [H, W, C]
    plt.show()

# 取出一个批次的图片和标签
dataiter = iter(trainloader)
images, labels = next(dataiter)
 
imshow(torchvision.utils.make_grid(images))  # 将一批图片拼成网格显示
# 将类别编号转换为类别名称
print('Labels: ', ' '.join('%5s ;' % classes[labels[j]] for j in range(batch_size)))

## Neural Network

###  Module

In [ ]:
# 定义卷积神经网络
class Net(nn.Module):
    def __init__(self):  # 创建网络中使用的层
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 3, padding=1)  # 3 通道输入、6 通道输出，使用 3×3 卷积核
        self.pool1 = nn.MaxPool2d(2, 2)  # 2×2 最大池化，将高和宽减半
        self.conv2 = nn.Conv2d(6, 16, 3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)

        self.relu1 = nn.ReLU()  # 将负数置为 0

        self.fc1 = nn.Linear(16 * 8 * 8, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)  # 输出 10 个类别的分数


    def forward(self, x):  # 定义数据依次经过哪些层
        x = self.conv1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.pool2(x)

        x = self.relu1(x)

        x = torch.flatten(x, 1)  # 保留批量维度，展平每张图片的特征
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

### Init the Net and check the model parameters 

In [ ]:
model = Net()  # 创建 Net 的实例，model 就是这个网络对象
# 遍历模型参数，查看各参数的名称和形状
for name, param in model.named_parameters():
    print(name, param.size())

### load pretrained models

In [ ]:
import torchvision.models as models
resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)  # 加载预训练的 ResNet18

for name, param in resnet18.named_parameters():
    print(name, param.size())

### 🎯 TASK-3
Create a separate copy of `Net` named `Task3Net`, and add a 2D batch normalization layer (`bn1`, 16 channels) before `relu1`. No execution is required.


### 🎯 TASK-4
Create a separate `Task4Net` based on Task 3. Update the layer definitions and forward pass:

1. Add a 3x3 convolutional layer (`conv3`) after `pool2`: 16 input channels, 64 output channels, stride 1, padding 1.
2. Add a 2x2 max-pooling layer (`pool3`, stride 2) after `conv3` and before `bn1`; change `bn1` to 64 channels.
3. Set `fc1` to 1024 input / 512 output features, and `fc2` to 512 input / 84 output features.

No execution is required. The following training section uses the original `Net` and `model`.


## Training

### Loss function and optimizer

In [ ]:
import torch.optim as optim

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)  # 模型和数据使用同一设备；先移动模型，再创建优化器
loss = nn.CrossEntropyLoss()  # 多分类任务使用的交叉熵损失
# optimizer = optim.SGD(model.parameters(), lr=0.01)
optimizer = optim.Adam(model.parameters(), lr=0.0001)  # 用 Adam 更新模型参数


### Training loop

In [ ]:
model.train()  # 设置训练模式；实际前向计算发生在 model(inputs) 时
epoch_num = 10
total_iter = len(trainloader)
for epoch in range(epoch_num):  # 每轮遍历一次训练集
    timestart = time.time()
    running_loss = 0.0
    total = 0
    correct = 0

    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()  # 清空上一批次的梯度，避免梯度累加

        outputs = model(inputs)  # 调用 Net.forward，每行是该图片的 10 个类别分数
        l = loss(outputs, labels)  # 用类别分数和真实标签计算交叉熵损失
        l.backward()  # 反向传播，计算梯度
        optimizer.step()  # 根据梯度更新模型参数

        # 每个批次都累计损失和正确数量；按样本数计算平均损失
        running_loss += l.item() * labels.size(0)
        total += labels.size(0)
        # dim=1 沿类别维找最高分，用最高分对应的类别作为预测。
        # max 返回（最大分数，类别编号）；_ 忽略分数，predicted 保留类别编号。
        _, predicted = torch.max(outputs.detach(), 1)
        correct += (predicted == labels).sum().item()

        if (i + 1) % 500 == 0 or (i + 1) == total_iter:
            print('Epoch %d/%d, batch %d/%d, loss: %.4f, accuracy: %.3f%% (%d images)' % (
                epoch + 1, epoch_num, i + 1, total_iter,
                running_loss / total, 100.0 * correct / total, total))
            running_loss = 0.0
            total = 0
            correct = 0

    print('epoch %d cost %.3f sec' % (epoch + 1, time.time() - timestart))

print('Finished Training')


## Save and load model

### Save model
Save a new checkpoint after training. The bundled `cnn_weight.tar` uses a different network structure, so this notebook saves to `cnn_checkpoint.pt`.


In [ ]:
from pathlib import Path

path = Path('../weight/cnn_checkpoint.pt')
path.parent.mkdir(parents=True, exist_ok=True)

# 保存本次训练的模型、优化器状态及损失数值
torch.save({'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': l.item()
            }, path)
print('Model saved to:', path)


### Load model and test
Run the training and saving cells first; the model structure must match the saved checkpoint.


In [ ]:
from tqdm import tqdm

checkpoint = torch.load(path, map_location=device, weights_only=True)  # 读取本次训练保存的检查点
model.load_state_dict(checkpoint['model_state_dict'])  # 恢复模型参数
print('Load weights from:', path)

def test(model, testloader, device='cpu'):
    model.to(device)
    model.eval()  # 切换到评估模式
    correct = 0
    total = 0
    with torch.no_grad():  # 测试时不计算梯度
        for images, labels in tqdm(testloader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)  # 调用模型的 forward，得到每张图片的类别分数
            _, predicted = torch.max(outputs, 1)  # 沿类别维取最高分；predicted 是类别编号
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print('Accuracy of the network on the %d test images: %.3f %%' % (
        total, 100.0 * correct / total))

test(model, testloader, device=device)


## Inspect intermediate features

In [ ]:
def hook_fn(module, inputs, output):
    print(f'Inside {module.__class__.__name__} forward hook')
    print('Input shape:', inputs[0].shape)
    print('Output shape:', output.shape)

hook_handle = model.fc2.register_forward_hook(hook_fn)  # 每次 fc2 前向计算后调用 hook_fn

model.eval()
images, _ = next(iter(testloader))  # 只用一个批次观察中间特征
try:
    with torch.no_grad():
        outputs = model(images.to(device))
finally:
    hook_handle.remove()  # 用完后移除钩子


### 🎯 TASK-5
Write training code for three-class image classification using the original `Net`. Assume a training data loader of 32x32 RGB images with labels 0, 1, and 2 is available; no dataset preparation or execution is required.

1. Modify `Net`'s `fc3` layer to output 3 class scores.
2. Choose an appropriate classification loss and optimizer.
3. Implement the training loop: forward propagation, loss calculation, gradient clearing, backpropagation, and parameter updates.
